# ABC Bank CLU - Slide Ready Report Pack

This notebook:
- reads the CLU evaluation outputs
- builds slide-ready summary tables
- creates concise commentary for a report or presentation
- exports presentation-ready CSV/TXT outputs

In [ ]:
# %pip install pandas matplotlib python-dotenv

In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv(".env")

PROJECT_NAME = os.getenv("AZURE_CLU_PROJECT_NAME", "abc-bank-clu")
DEPLOYMENT_NAME = os.getenv("AZURE_CLU_DEPLOYMENT_NAME", "production")

BASE_DIR = Path("analysis_output")
SUMMARY_PATH = BASE_DIR / "abc_bank_clu_evaluation_summary.csv"
PER_INTENT_PATH = BASE_DIR / "abc_bank_clu_per_intent_summary.csv"
FAILED_CASES_PATH = BASE_DIR / "abc_bank_clu_failed_cases.csv"
DETAILED_PATH = BASE_DIR / "abc_bank_clu_detailed_results.csv"

for p in [SUMMARY_PATH, PER_INTENT_PATH, DETAILED_PATH]:
    if not p.exists():
        raise FileNotFoundError(
            f"Required evaluation file not found: {p.resolve()}\n"
            "Run the evaluation notebook first."
        )

print("Project:", PROJECT_NAME)
print("Deployment:", DEPLOYMENT_NAME)
print("Reading evaluation outputs from:", BASE_DIR.resolve())

In [ ]:
# =========================
# LOAD INPUTS
# =========================
df_summary = pd.read_csv(SUMMARY_PATH)
df_per_intent = pd.read_csv(PER_INTENT_PATH)
df_detailed = pd.read_csv(DETAILED_PATH)

if FAILED_CASES_PATH.exists():
    df_failed = pd.read_csv(FAILED_CASES_PATH)
else:
    df_failed = pd.DataFrame()

print("Summary rows:", len(df_summary))
print("Per-intent rows:", len(df_per_intent))
print("Detailed rows:", len(df_detailed))
print("Failed-case rows:", len(df_failed))

In [ ]:
# =========================
# CORE VALUES
# =========================
summary = df_summary.iloc[0].to_dict()

total_tests = int(summary["total_tests"])
overall_pass_rate = float(summary["overall_pass_rate_pct"])
intent_pass_rate = float(summary["intent_pass_rate_pct"])
entity_category_pass_rate = float(summary["entity_category_pass_rate_pct"])
entity_text_pass_rate = float(summary["entity_text_pass_rate_pct"])
average_confidence = float(summary["average_top_intent_score"])

best_intent_row = df_per_intent.sort_values(
    ["overall_pass_rate_pct", "avg_top_intent_score"],
    ascending=[False, False]
).iloc[0]

worst_intent_row = df_per_intent.sort_values(
    ["overall_pass_rate_pct", "avg_top_intent_score"],
    ascending=[True, True]
).iloc[0]

best_intent = best_intent_row["expected_intent"]
best_intent_pass = float(best_intent_row["overall_pass_rate_pct"])

worst_intent = worst_intent_row["expected_intent"]
worst_intent_pass = float(worst_intent_row["overall_pass_rate_pct"])

print("Overall pass rate:", overall_pass_rate)
print("Best intent:", best_intent, best_intent_pass)
print("Worst intent:", worst_intent, worst_intent_pass)

In [ ]:
# =========================
# SLIDE-READY TABLE 1
# EXECUTIVE KPI TABLE
# =========================
kpi_table = pd.DataFrame([
    ["Project Name", PROJECT_NAME],
    ["Deployment Name", DEPLOYMENT_NAME],
    ["Total Test Cases", total_tests],
    ["Overall Pass Rate (%)", round(overall_pass_rate, 2)],
    ["Intent Pass Rate (%)", round(intent_pass_rate, 2)],
    ["Entity Category Pass Rate (%)", round(entity_category_pass_rate, 2)],
    ["Entity Text Pass Rate (%)", round(entity_text_pass_rate, 2)],
    ["Average Top Intent Confidence", round(average_confidence, 4)],
])

kpi_table.columns = ["Metric", "Value"]
kpi_table

In [ ]:
# =========================
# SLIDE-READY TABLE 2
# INTENT PERFORMANCE TABLE
# =========================
intent_table = df_per_intent.copy()

intent_table = intent_table.rename(columns={
    "expected_intent": "Intent",
    "total_tests": "Total Tests",
    "intent_pass_count": "Intent Pass Count",
    "overall_pass_count": "Overall Pass Count",
    "avg_top_intent_score": "Avg Confidence",
    "intent_pass_rate_pct": "Intent Pass Rate (%)",
    "overall_pass_rate_pct": "Overall Pass Rate (%)",
})

intent_table = intent_table[
    [
        "Intent",
        "Total Tests",
        "Intent Pass Count",
        "Overall Pass Count",
        "Intent Pass Rate (%)",
        "Overall Pass Rate (%)",
        "Avg Confidence",
    ]
]

intent_table

In [ ]:
# =========================
# SLIDE-READY TABLE 3
# FAILED CASES TABLE
# =========================
if df_failed.empty:
    failed_slide_table = pd.DataFrame(columns=[
        "Utterance",
        "Expected Intent",
        "Actual Intent",
        "Top Intent Score",
        "Issue Summary"
    ])
else:
    failed_work = df_detailed.loc[~df_detailed["overall_pass"]].copy()

    failed_slide_table = failed_work.rename(columns={
        "utterance": "Utterance",
        "expected_intent": "Expected Intent",
        "actual_top_intent": "Actual Intent",
        "top_intent_score": "Top Intent Score",
        "failure_reason": "Issue Summary",
    })[
        ["Utterance", "Expected Intent", "Actual Intent", "Top Intent Score", "Issue Summary"]
    ]

failed_slide_table

In [ ]:
# =========================
# COMMENTARY BLOCKS
# =========================
def build_executive_commentary():
    lines = []
    lines.append(
        f"The ABC Bank CLU model was evaluated on {total_tests} test cases across three business intents: "
        f"CheckBalance, TransferFunds, and LoanStatus."
    )
    lines.append(
        f"The model achieved an overall pass rate of {overall_pass_rate:.2f}% with an average top-intent confidence "
        f"score of {average_confidence:.4f}."
    )
    lines.append(
        f"Intent classification performed at {intent_pass_rate:.2f}%, while entity category detection reached "
        f"{entity_category_pass_rate:.2f}% and entity text extraction reached {entity_text_pass_rate:.2f}%."
    )
    lines.append(
        f"The strongest-performing intent was {best_intent} at {best_intent_pass:.2f}% overall pass rate, "
        f"while {worst_intent} was the weakest at {worst_intent_pass:.2f}%."
    )
    if len(failed_slide_table) == 0:
        lines.append(
            "No failed cases were observed in the current validation set, indicating stable intent and entity behavior "
            "for the tested banking scenarios."
        )
    else:
        lines.append(
            f"{len(failed_slide_table)} failed cases remain, which indicates targeted improvement is still needed for "
            "selected utterance patterns, entity spans, or ambiguous phrasing."
        )
    return " ".join(lines)

def build_findings_commentary():
    lines = []
    lines.append(
        "The current CLU model demonstrates strong readiness for controlled banking assistant scenarios, especially "
        "where user phrasing closely matches the training examples."
    )
    if overall_pass_rate >= 90:
        lines.append(
            "Overall quality is strong enough for a lab or prototype deployment, with only limited corrective tuning required."
        )
    elif overall_pass_rate >= 75:
        lines.append(
            "Model quality is moderate and usable for demonstration purposes, but more utterance coverage is needed before production use."
        )
    else:
        lines.append(
            "Model quality is still immature and requires broader training data coverage before it can be trusted in front-end scenarios."
        )

    if entity_text_pass_rate < entity_category_pass_rate:
        lines.append(
            "The gap between entity category detection and entity text extraction suggests that span labeling consistency should be improved."
        )

    if len(failed_slide_table) > 0:
        most_common_issue = (
            failed_slide_table["Issue Summary"].astype(str).value_counts().idxmax()
        )
        lines.append(
            f"The most common failure pattern in the current test set is: {most_common_issue}."
        )
    return " ".join(lines)

def build_recommendations_commentary():
    recs = []
    recs.append(
        "Expand utterance coverage for TransferFunds and any lower-performing intent using more paraphrases, shorter variants, and realistic user wording."
    )
    recs.append(
        "Add more labeled examples for account_type and amount to improve entity span detection, especially for short account references such as 'savings' or 'loan account'."
    )
    recs.append(
        "Introduce harder negative and edge-case utterances to test ambiguity, missing values, and multi-entity phrasing."
    )
    recs.append(
        "Re-train and re-run the evaluation pack after every major update so that intent accuracy, entity detection, and confidence trends remain measurable."
    )
    return recs

executive_commentary = build_executive_commentary()
findings_commentary = build_findings_commentary()
recommendations_commentary = build_recommendations_commentary()

print("EXECUTIVE COMMENTARY")
print(executive_commentary)
print("\nFINDINGS COMMENTARY")
print(findings_commentary)
print("\nRECOMMENDATIONS")
for i, rec in enumerate(recommendations_commentary, start=1):
    print(f"{i}. {rec}")

In [ ]:
# =========================
# SLIDE TITLES + TALK TRACKS
# =========================
slide_pack = pd.DataFrame([
    {
        "Slide Title": "ABC Bank CLU Evaluation Overview",
        "Purpose": "Summarize overall model quality and key KPIs",
        "Suggested Content": "Use KPI table and overall pass/fail chart",
        "Speaker Notes": executive_commentary,
    },
    {
        "Slide Title": "Intent-Level Performance",
        "Purpose": "Show how each intent performed",
        "Suggested Content": "Use intent performance table and pass-rate/confidence charts",
        "Speaker Notes": findings_commentary,
    },
    {
        "Slide Title": "Failure Analysis and Improvement Priorities",
        "Purpose": "Highlight failed cases and tuning focus areas",
        "Suggested Content": "Use failed cases table and recommendation list",
        "Speaker Notes": " ".join(recommendations_commentary),
    },
])

slide_pack

In [ ]:
# =========================
# CHART - PRESENTATION READY 1
# OVERALL KPI BAR
# =========================
kpi_chart = pd.DataFrame({
    "Metric": [
        "Overall Pass Rate",
        "Intent Pass Rate",
        "Entity Category Pass Rate",
        "Entity Text Pass Rate"
    ],
    "Value": [
        overall_pass_rate,
        intent_pass_rate,
        entity_category_pass_rate,
        entity_text_pass_rate
    ]
})

plt.figure(figsize=(8, 4.5))
plt.bar(kpi_chart["Metric"], kpi_chart["Value"])
plt.title("ABC Bank CLU Evaluation KPIs")
plt.xlabel("Metric")
plt.ylabel("Percentage")
plt.ylim(0, 100)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# CHART - PRESENTATION READY 2
# OVERALL PASS RATE BY INTENT
# =========================
plt.figure(figsize=(8, 4.5))
plt.bar(intent_table["Intent"], intent_table["Overall Pass Rate (%)"])
plt.title("Overall Pass Rate by Intent")
plt.xlabel("Intent")
plt.ylabel("Pass Rate (%)")
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# CHART - PRESENTATION READY 3
# AVERAGE CONFIDENCE BY INTENT
# =========================
plt.figure(figsize=(8, 4.5))
plt.bar(intent_table["Intent"], intent_table["Avg Confidence"])
plt.title("Average Confidence by Intent")
plt.xlabel("Intent")
plt.ylabel("Confidence Score")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# OPTIONAL EXPORTS
# =========================
report_dir = Path("analysis_output") / "report_pack"
report_dir.mkdir(parents=True, exist_ok=True)

kpi_table_path = report_dir / "slide_kpi_table.csv"
intent_table_path = report_dir / "slide_intent_table.csv"
failed_table_path = report_dir / "slide_failed_cases_table.csv"
slide_pack_path = report_dir / "slide_pack.csv"
commentary_path = report_dir / "slide_commentary.txt"

kpi_table.to_csv(kpi_table_path, index=False, encoding="utf-8-sig")
intent_table.to_csv(intent_table_path, index=False, encoding="utf-8-sig")
failed_slide_table.to_csv(failed_table_path, index=False, encoding="utf-8-sig")
slide_pack.to_csv(slide_pack_path, index=False, encoding="utf-8-sig")

with open(commentary_path, "w", encoding="utf-8") as f:
    f.write("EXECUTIVE COMMENTARY\n")
    f.write(executive_commentary + "\n\n")
    f.write("FINDINGS COMMENTARY\n")
    f.write(findings_commentary + "\n\n")
    f.write("RECOMMENDATIONS\n")
    for i, rec in enumerate(recommendations_commentary, start=1):
        f.write(f"{i}. {rec}\n")

print("Saved:")
print(kpi_table_path.resolve())
print(intent_table_path.resolve())
print(failed_table_path.resolve())
print(slide_pack_path.resolve())
print(commentary_path.resolve())